<a href="https://colab.research.google.com/github/AI4Finance-Foundation/FinGPT/blob/master/FinGPT_Training_LoRA_with_ChatGLM2_6B_for_Beginners.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Getting Started with FinGPT
Welcome to this comprehensive guide aimed at beginners diving into the realm of Financial Large Language Models (FinLLMs) with FinGPT. This blog post demystifies the process of training FinGPT using Low-Rank Adaptation (LoRA) with the robust base model ChatGlm2-6b.



## Part 1: Prepare the financial sentiment dataset


In [ ]:
%pip install "transformers==4.40.2" "peft==0.10.0" "accelerate==0.29.3" "bitsandbytes==0.45.5" "datasets==2.19.2" "pyarrow<21" sentencepiece cpm_kernels ipywidgets


### 1.1 Select a CUDA runtime

Use an NVIDIA CUDA-enabled Colab runtime or a Linux/WSL2 Jupyter kernel with CUDA-enabled PyTorch 2.3 or newer already installed. Run the install cell once, restart the kernel, then continue here. Begin with a single GPU and a batch size of one. Peak VRAM depends on the sequence length, GPU runtime, and installed libraries. The defaults are intended for a Colab T4-class GPU (about 16 GB); start with the smoke run below to check your runtime.


In [ ]:
from pathlib import Path

import torch

if not torch.cuda.is_available():
    raise RuntimeError("Select an NVIDIA GPU runtime or a CUDA-enabled local PyTorch kernel.")

print(torch.cuda.get_device_name())
print(f'GPU memory: {torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GiB')
smoke_run = False  # Set True for a one-step check before a full run.
jsonl_path = Path("chatglm2_lora_data/dataset.jsonl")
jsonl_path.parent.mkdir(exist_ok=True)


### 1.2 Load and Prepare Dataset:

* Import necessary libraries from the datasets package: https://huggingface.co/docs/datasets/index
* Load the Twitter Financial News Sentiment (TFNS) dataset and convert it to a Pandas dataframe. https://huggingface.co/datasets/zeroshot/twitter-financial-news-sentiment
* Map numerical labels to their corresponding sentiments (negative, positive, neutral).
* Add instruction for each data entry, which is crucial for Instruction Tuning.
* Convert the Pandas dataframe back to a Hugging Face Dataset object.




In [ ]:
from datasets import load_dataset
import datasets

dic = {
    0:"negative",
    1:'positive',
    2:'neutral',
}

tfns = load_dataset('zeroshot/twitter-financial-news-sentiment')
tfns = tfns['train']
tfns = tfns.to_pandas()
tfns['label'] = tfns['label'].apply(lambda x:dic[x])
tfns['instruction'] = 'What is the sentiment of this tweet? Please choose an answer from {negative/neutral/positive}.'
tfns.columns = ['input', 'output', 'instruction']
tfns = datasets.Dataset.from_pandas(tfns)
tfns

### 1.3 Shuffle the dataset


In [ ]:
all_dataset = tfns.shuffle(seed=42)
if smoke_run:
    all_dataset = all_dataset.select(range(64))
all_dataset.shape


Now that your training data is loaded and prepared.

## Part 2: Dataset Formatting and Tokenization
Once your data is prepared, the next steps involve formatting the dataset for model ingestion and tokenizing the input data. Below, we provide a step-by-step breakdown of the code snippets shared.



### 2.1 Dataset Formatting:
You need to structure your data in a specific format that aligns with the training process.



In [ ]:
import json
from tqdm.notebook import tqdm

In [ ]:
def format_example(example: dict) -> dict:
    context = f"Instruction: {example['instruction']}\n"
    if example.get("input"):
        context += f"Input: {example['input']}\n"
    context += "Answer: "
    target = example["output"]
    return {"context": context, "target": target}

In [ ]:
data_list = []
for item in all_dataset.to_pandas().itertuples():
    tmp = {}
    tmp["instruction"] = item.instruction
    tmp["input"] = item.input
    tmp["output"] = item.output
    data_list.append(tmp)

In [ ]:
# save to a jsonl file
with open(jsonl_path, 'w') as f:
    for example in tqdm(data_list, desc="formatting.."):
        f.write(json.dumps(format_example(example)) + '\n')

### 2.2 Tokenization
Tokenization is the process of converting input text into tokens that can be fed into the model.



In [ ]:
from transformers import AutoConfig, AutoTokenizer

model_name = "THUDM/chatglm2-6b"
max_seq_length = 256
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True, use_fast=False)
config = AutoConfig.from_pretrained(model_name, trust_remote_code=True)


In [ ]:
def preprocess(example):
    target_ids = tokenizer.encode(
        example["target"], add_special_tokens=False, max_length=16, truncation=True
    ) + [config.eos_token_id]
    prompt_ids = tokenizer.encode(
        example["context"], max_length=max_seq_length - len(target_ids), truncation=True
    )
    return {"input_ids": prompt_ids + target_ids, "seq_len": len(prompt_ids)}


### 2.3 Tokenize the formatted dataset


In [ ]:
dataset = load_dataset("json", data_files=str(jsonl_path), split="train")
dataset = dataset.map(preprocess, remove_columns=dataset.column_names)


## Part 3: Configure ChatGLM2 LoRA training


### 3.1 Training Arguments Setup:
Initialize and set training arguments.



In [ ]:
from transformers import AutoModel, BitsAndBytesConfig, Trainer, TrainingArguments
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training


In [ ]:
training_args = TrainingArguments(
    output_dir="./finetuned_model_smoke" if smoke_run else "./finetuned_model",
    max_steps=1 if smoke_run else -1,
    num_train_epochs=2,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=1 if smoke_run else 32,
    learning_rate=1e-4,
    logging_steps=1 if smoke_run else 50,
    evaluation_strategy="no" if smoke_run else "epoch",
    save_strategy="no" if smoke_run else "epoch",
    load_best_model_at_end=not smoke_run,
    remove_unused_columns=False,
    report_to="none",
)


### 3.2 Quantization Config Setup:
Set quantization configuration to reduce model size without losing significant precision.



In [ ]:
 # Quantization
q_config = BitsAndBytesConfig(load_in_4bit=True,
                                bnb_4bit_quant_type='nf4',
                                bnb_4bit_use_double_quant=True,
                                bnb_4bit_compute_dtype=torch.float16
                                )

### 3.3 Load and prepare the 4-bit base model


In [ ]:
model = AutoModel.from_pretrained(
    model_name, quantization_config=q_config, trust_remote_code=True, device_map={"": 0}
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)


### 3.4 LoRA Config & Setup:
Implement Low-Rank Adaptation (LoRA) and print trainable parameters.



In [ ]:
# LoRA
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["query_key_value"],
    bias='none',
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Part 4: Split the data and train the adapter


### 4.1 Loading Your Data:


In [ ]:
dataset = dataset.train_test_split(test_size=0.2, seed=42)


### 4.2 Train with dynamic padding and mask prompt and padding tokens from the loss


In [ ]:
def data_collator(features):
    longest = max(len(feature["input_ids"]) for feature in features)
    input_ids, labels, attention_mask = [], [], []
    for feature in features:
        ids = feature["input_ids"]
        prompt_length = feature["seq_len"]
        padding = longest - len(ids)
        input_ids.append(ids + [tokenizer.pad_token_id] * padding)
        labels.append([-100] * prompt_length + ids[prompt_length:] + [-100] * padding)
        attention_mask.append([1] * len(ids) + [0] * padding)
    return {
        "input_ids": torch.tensor(input_ids),
        "labels": torch.tensor(labels),
        "attention_mask": torch.tensor(attention_mask),
    }


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    data_collator=data_collator,
)
trainer.train()
trainer.save_model()


### 4.3 Find the saved adapter

The adapter is saved to the path printed below. In Colab you can download its files from the Files pane.


In [ ]:
print(Path(training_args.output_dir).resolve())


## Part 5: Inference and optional benchmarks

Reuse the trained model in memory. Run the benchmark cells after a full training run.


### 5.1 Try the trained adapter


In [ ]:
model.eval()
model.config.use_cache = True
example = format_example({
    "instruction": "What is the sentiment of this tweet? Choose negative, neutral, or positive.",
    "input": "The company reported record profits.",
    "output": "",
})
inputs = tokenizer(example["context"], return_tensors="pt").to(model.device)
with torch.inference_mode():
    output = model.generate(**inputs, max_new_tokens=8, do_sample=False)
print(tokenizer.decode(output[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True))


In [ ]:
import subprocess
import sys

finnlp_path = Path("FinNLP").resolve()
if not finnlp_path.exists():
    subprocess.run(["git", "clone", "https://github.com/AI4Finance-Foundation/FinNLP.git"], check=True)
sys.path.insert(0, str(finnlp_path))


In [ ]:
from transformers import AutoModel, AutoTokenizer, AutoModelForCausalLM

from peft import PeftModel
import torch

# Load benchmark datasets from FinNLP
from finnlp.benchmarks.fpb import test_fpb
from finnlp.benchmarks.fiqa import test_fiqa , add_instructions
from finnlp.benchmarks.tfns import test_tfns
from finnlp.benchmarks.nwgi import test_nwgi

### 5.2 Optional FinNLP benchmarks


In [ ]:
batch_size = 1


In [ ]:
# TFNS Test Set, len 2388
# Available: 84.85 compute units
res = test_tfns(model, tokenizer, batch_size = batch_size)
# Available: 83.75 compute units
# Took about 1 compute unite to inference

In [ ]:
# FPB, len 1212
res = test_fpb(model, tokenizer, batch_size = batch_size)

In [ ]:
# FiQA, len 275
res = test_fiqa(model, tokenizer, prompt_fun = add_instructions, batch_size = batch_size)

In [ ]:
# NWGI, len 4047
res = test_nwgi(model, tokenizer, batch_size = batch_size)

## Troubleshooting

If bitsandbytes or Accelerate is reported missing after installation, restart the kernel and confirm that it is using the same Python environment as the install cell. If CUDA runs out of memory, restart the kernel to release old model instances and lower the sequence length before rerunning preprocessing.
